# NeurIPS 2026 Smart Buildings — Version 25: soft CBF-QP safety filter

**Goal:** Preserve Version 7's efficient physics-based HVAC dispatch while reducing severe temperature excursions.

### Components

1. PhysicsBaselinePolicy: the original V7 controller, copied verbatim apart from the class name. It owns AHU on/off, coil and economizer operation, initial VAV commands, fan pressure, and hot-water dispatch.
2. TransferPolicy: one-step zone temperature-rate estimator and *soft* discrete-time temperature barrier.
3. Local QP: modifies **cooling dampers** or **reheat valves** when barrier risk is positive; never assumes that increasing the VAV damper provides heating.
4. Optional fan QP: increases shared AHU pressure only for an *overheated* zone with positive coil cooling headroom, a nearly saturated VAV, and residual barrier demand.

### Barrier and objective

The upper and lower comfort barriers are h_upper = upper_setpoint - 0.20 K - T and h_lower = T - lower_setpoint - 0.20 K. The proxy one-step inequality is h_next >= (1 - 0.28) * h_now. For a positive constraint deficit b (Kelvin), solve:

    minimize 0.5 * sum(weight_i * correction_i^2) + 0.5 * penalty * slack^2
    subject to sum(gain_i * correction_i) + slack >= b
               0 <= correction_i <= cap_i, slack >= 0

The QP is convex and always feasible because slack is allowed. It uses 26 fixed dual bisection iterations and NumPy only; runtime is O(number of AHUs + zones) per policy step for fixed iteration count.

### Preserving V7 efficiency

- The V7 nominal policy is not altered; equipment run state, coil setpoint, economizer, and shared hot-water commands remain V7's.
- A mild zone correction is bounded to +0.08 physical actuator fraction; a severe one to +0.25.
- Heating corrections are reheat-valve only; increasing airflow could have the *wrong thermal sign* because the AHU is cooling-only.
- Fan correction is at most +0.18 in normalized pressure-action units and occurs only for severe overheating with a near-saturated VAV and positive coil cooling headroom.
- When no barrier is active, the returned action remains bitwise identical to V7's.

### Limitations and evaluation

**No certified safety guarantee:** zone drift and actuator gains are uncalibrated proxy dynamics. True delivered-air temperature, HVAC delay, actuator lag, and weather disturbances can invalidate a predicted safety improvement. Soft slack also permits violations. Compare real temperature violations and energy in the organizer simulator before deciding whether V25 improves on V7.

Run offline checks: python Co_Kaggle/g8/submissions/test_version25.py

Submit submission-nb - Version 25.ipynb to Kaggle. The last cell invokes the official SmartBuildingsInferenceServer with TransferPolicy. Compare V25's unscored results with V7's recorded public loss of **0.03947**.

**Controlled comparison:** V7 outdoor-air decoding is retained (including its known normalization approximation) rather than silently changed; this isolates the incremental safety-filter experiment.


In [ ]:
"""Version 7: physics-based, load-matched HVAC dispatch.

Distinct experiment: use the AHU mixing equation and estimated delivered
supply-air temperature to size airflow and avoid unnecessary compressor work.
This is feed-forward engineering control, not PID, temperature prediction,
risk-weighted allocation, RLS, or candidate-action optimization.

Runtime: O(number of AHUs + number of zones) per five-minute step.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class PhysicsBaselinePolicy(smart_buildings_inference_server.Policy):
    """Weather-compensated, thermodynamics-aware air and hot-water dispatch.

    Action contract: continuous float32 in [-1, 1], discrete int32 in {0, 1}.
    All equipment topology and sensor locations come from episode metadata.
    """

    # Physical constants from the starter-kit action descriptions.
    MIN_SUPPLY_K = 285.0
    MAX_SUPPLY_K = 305.0
    MAX_RETURN_FRACTION = 0.70
    ACTIVE_THRESHOLD_K = 0.18
    MIN_VAV_FRACTION = 0.10
    MIN_FAN_PRESSURE_PA = 200.0
    # A cooling-only AHU still admits at least 30% outside air.
    # Increasing intake beyond that floor also raises exhaust-fan power.
    MIN_ECONOMIZER_OUTDOOR_K = 287.0
    MIN_ECONOMIZER_RECIRC = 0.15

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
                "outdoor": self.fields.get(
                    f"ahs_{identifier}_outside_air_temperature_sensor"),
            })
        self.reset()

    def reset(self):
        """No episode state: decisions depend only on the current observation."""
        return None

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _schedule_fallback(self, obs):
        """Fall back to the occupied/eco schedule if zone setpoints are absent."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "actions": zone,
            "temp": temperature,
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _select_air_supply(self, zones, outside_k):
        """Select coil setpoint and air mixing from the plant's heat balance.

        Returns (coil_setpoint_K, return_fraction, delivered_K).

        The cooling coil can only LOWER the mixed-air temperature.
        Economizer mode is enabled only if extra outside air can entirely
        avoid mechanical cooling. Otherwise the guaranteed extra exhaust
        airflow can erase a marginal chiller saving; stay at the 70%
        return-air limit until that trade-off has been measured.
        """
        hottest = max(zone["hot"] for zone in zones)
        return_k = float(np.mean([zone["temp"] for zone in zones]))

        if hottest < self.ACTIVE_THRESHOLD_K:
            coil_setpoint_k = self.MAX_SUPPLY_K
        else:
            hottest_zones = [zone["temp"] for zone in zones
                             if zone["hot"] > 0.0]
            mean_hot_k = float(np.mean(hottest_zones))
            coil_setpoint_k = self._clamp(
                mean_hot_k - 4.0 - 1.3 * hottest,
                self.MIN_SUPPLY_K, 297.0)

        if not np.isfinite(outside_k):
            outside_k = return_k

        return_fraction = self.MAX_RETURN_FRACTION
        if (hottest >= self.ACTIVE_THRESHOLD_K
                and self.MIN_ECONOMIZER_OUTDOOR_K <= outside_k < return_k - 1.5):
            temperature_span = return_k - outside_k
            # The lowest mix reachable at 15% return air must be cool
            # enough to meet the coil setpoint without compressor work.
            min_mix_k = (self.MIN_ECONOMIZER_RECIRC * return_k
                         + (1.0 - self.MIN_ECONOMIZER_RECIRC) * outside_k)
            if min_mix_k <= coil_setpoint_k:
                required_return = (coil_setpoint_k - outside_k) / temperature_span
                return_fraction = self._clamp(
                    required_return,
                    self.MIN_ECONOMIZER_RECIRC, self.MAX_RETURN_FRACTION)

        mixed_k = (return_fraction * return_k
                   + (1.0 - return_fraction) * outside_k)
        delivered_k = min(mixed_k, coil_setpoint_k)
        return coil_setpoint_k, return_fraction, delivered_k

    def _control_zone_boxes(self, statuses, delivered, hot_water_demands, continuous):
        """Size VAV flow using actual heating/cooling temperature headroom.

        In contrast to a fixed damper = gain * temperature-error rule,
        requested damper opening falls as cooling headroom grows. This is
        a *proxy* for delivered flow, not a calibrated CFM calculation.
        Returns (mean_damper, peak_damper) for AHU fan coordination.
        """
        sum_damper = 0.0
        peak_damper = 0.0
        for status in statuses:
            zone = status["actions"]
            hot, cold, temperature = (
                status["hot"], status["cold"], status["temp"])
            if hot >= self.ACTIVE_THRESHOLD_K:
                headroom = max(2.5, temperature - delivered)
                damper = self._clamp(0.12 + 2.1 * hot / headroom,
                                     self.MIN_VAV_FRACTION, 0.98)
            elif cold >= self.ACTIVE_THRESHOLD_K:
                # Heating is VAV reheat, never the AHU cooling coil.
                damper = self._clamp(0.22 + 0.21 * cold, 0.18, 0.88)
                if zone["valve"] is not None:
                    valve = self._clamp(0.16 + 0.31 * cold, 0.0, 1.0)
                    continuous[zone["valve"]] = self._fraction_action(valve)
                    hot_water_demands.append((cold, valve))
            else:
                damper = self.MIN_VAV_FRACTION
            sum_damper += damper
            peak_damper = max(peak_damper, damper)
            if zone["damper"] is not None:
                continuous[zone["damper"]] = self._fraction_action(damper)
        # Position fractions are proxies, not calibrated volumetric flow.
        return sum_damper / len(statuses), peak_damper

    def _control_ahu(self, ahu, statuses, outdoor, continuous, discrete, water):
        """Run only if a zone needs conditioning; never call the AHU to heat."""
        if not statuses:
            return
        worst_violation_k = max(max(z["hot"], z["cold"]) for z in statuses)
        if worst_violation_k < self.ACTIVE_THRESHOLD_K:
            return
        target, recirc, delivered = self._select_air_supply(statuses, outdoor)
        mean_damper, peak_damper = self._control_zone_boxes(
            statuses, delivered, water, continuous)
        discrete[ahu["run"]] = 1
        continuous[ahu["supply"]] = self._temperature_action(target)
        if ahu["recirc"] is not None:
            continuous[ahu["recirc"]] = self._fraction_action(recirc)
        if ahu["pressure"] is not None:
            # Use both typical load and worst VAV. A badly overheated room
            # must not disappear in the average of many neutral zones.
            # The fan model uses airflow ~ sqrt(static pressure).
            demand_proxy = max(mean_damper, 0.50 * peak_damper)
            pressure_pa = self.MIN_FAN_PRESSURE_PA + 1250.0 * np.sqrt(
                self._clamp(
                    (demand_proxy - self.MIN_VAV_FRACTION) / 0.65,
                    0.0, 1.0))
            continuous[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)

    def _control_hot_water(self, water, outdoor_values, continuous, discrete):
        """Weather-compensated hot-water reset, independent of cooling mode.

        Cold outdoor air forces a lower mixed-air temperature because the
        simulator caps recirculation at 70%; compensate with hotter reheat
        water. Only start the pump if a VAV actually requests heating.
        """
        if not water:
            return
        if 0 <= self.water_run < self.n_discrete:
            discrete[self.water_run] = 1
        else:
            return
        worst = max(shortfall for shortfall, _ in water)
        average_valve = sum(valve for _, valve in water) / len(water)
        outside = (sum(outdoor_values) / len(outdoor_values)
                   if outdoor_values else 294.0)
        water_kelvin = self._clamp(
            318.0 + 0.35 * max(0.0, 294.0 - outside) + 5.0 * worst,
            320.0, 349.0)
        if self.water_supply is not None:
            continuous[self.water_supply] = self._fraction_action(
                (water_kelvin - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._fraction_action(
                self._clamp(0.38 + 0.52 * average_valve, 0.38, 0.95))

    def get_action(self, obs):
        """Read zones -> plan AHU air -> set VAVs -> coordinate shared water.

        Performs one pass per AHU and zone; no search, learned model, or
        state carried between five-minute decisions.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        hot_water_demands, outdoors = [], []
        for ahu in self.ahus:
            statuses = [
                self._read_zone(obs, z, fallback) for z in ahu["zones"]
            ]
            statuses = [s for s in statuses if s is not None]
            outdoor = self._value(obs, ahu["outdoor"])
            if np.isfinite(outdoor) and 210.0 <= outdoor <= 330.0:
                outdoors.append(outdoor)
            else:
                outdoor = float("nan")
            self._control_ahu(
                ahu, statuses, outdoor, continuous, discrete, hot_water_demands)
        self._control_hot_water(hot_water_demands, outdoors, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {"continuous_action": continuous, "discrete_action": discrete}


class TransferPolicy(PhysicsBaselinePolicy):
    """Apply small, soft-CBF safety corrections after V7 makes its decision.

    The V7 baseline owns equipment switching, cooling-coil settings, fan
    dispatch, economizer operation, and the shared hot-water system.
    This wrapper adjusts only active cooling dampers, reheat valves, and
    (as a last resort) the fan pressure of an overheated AHU.

    A CBF normally requires a verified dynamics model. Our one-step thermal
    sensitivities are *proxies*, so these soft QPs reduce modeled risk but
    do not certify the actual building's temperature safety.
    """

    # Per-step discrete barrier: h_next >= (1-alpha) * h_now.
    BARRIER_ALPHA = 0.28
    GUARD_BAND_K = 0.20
    NEAR_BOUNDARY_K = 0.40
    SEVERE_EXCURSION_K = 0.55

    # Corrective commands are nonnegative to preserve the V7 operating point.
    MILD_FRACTION_CAP = 0.08
    SEVERE_FRACTION_CAP = 0.25
    MAX_FAN_ACTION_CORRECTION = 0.18
    MAX_COMMANDED_FRACTION = 0.98
    MIN_COOLING_HEADROOM_K = 1.0
    FAN_DAMPER_THRESHOLD = 0.85

    # Conservative, uncalibrated thermal authority proxies (K per step).
    DAMPER_GAIN_PER_K = 0.011
    MAX_COOLING_GAIN = 0.17
    REHEAT_VALVE_GAIN = 0.125
    FAN_GAIN = 0.065

    MAX_RATE_K_PER_STEP = 0.30
    MAX_VALID_STEP_K = 1.50
    QP_BISECTION_STEPS = 26

    def reset(self):
        """Clear the rate estimator whenever Kaggle starts a new episode."""
        super().reset()
        self._zone_history = {}

    @staticmethod
    def _valid_index(index, length):
        return isinstance(index, (int, np.integer)) and 0 <= index < length

    @staticmethod
    def _fraction(continuous, index):
        """Convert a normalized [-1, 1] VAV action to opening in [0, 1]."""
        return 0.5 * (float(continuous[index]) + 1.0)

    def _increase_fraction(self, continuous, index, correction):
        """Write only actual corrections, avoiding no-op float32 roundtrips."""
        if correction > 0.0:
            opening = self._fraction(continuous, index)
            continuous[index] = self._fraction_action(opening + correction)

    def _zone_bounds(self, obs, zone, fallback):
        """Use the exact setpoint validity checks from V7."""
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            return fallback
        return low, high

    def _temperature_rate(self, sensor_index, temperature):
        """Estimate temperature drift in K per five-minute policy step.

        Limit both the observed step jump and the retained filtered rate;
        isolated sensor discontinuities start a fresh rate estimate.
        """
        previous = self._zone_history.get(sensor_index)
        rate = 0.0
        if previous is not None:
            step = temperature - previous[0]
            if abs(step) <= self.MAX_VALID_STEP_K:
                rate = (
                    0.45 * previous[1]
                    + 0.55 * self._clamp(
                        step, -self.MAX_RATE_K_PER_STEP,
                        self.MAX_RATE_K_PER_STEP,
                    )
                )
        self._zone_history[sensor_index] = (temperature, rate)
        return rate

    @classmethod
    def _solve_soft_cbf_qp(cls, deficit, gains, caps, weights, penalty):
        """Minimize a weighted control change subject to a soft barrier.

        min  0.5 * sum(w_i * d_i**2) + 0.5 * penalty * slack**2
        s.t. sum(g_i * d_i) + slack >= deficit
             0 <= d_i <= cap_i, slack >= 0

        With positive gains/weights, the KKT conditions yield
            d_i = clip(lambda * g_i / w_i, 0, cap_i)
            slack = lambda / penalty.

        Bisection finds the shared nonnegative dual multiplier. The upper
        bracket (penalty * deficit) is always feasible via slack alone.
        This avoids a generic QP solver in the Kaggle inference loop.
        """
        if not (len(gains) == len(caps) == len(weights)):
            raise ValueError("QP coefficient arrays must have equal length")
        if deficit <= 0.0:
            return [0.0] * len(gains), 0.0

        lo, hi = 0.0, penalty * deficit
        for _ in range(cls.QP_BISECTION_STEPS):
            multiplier = 0.5 * (lo + hi)
            improvement = multiplier / penalty
            for gain, cap, weight in zip(gains, caps, weights):
                improvement += gain * min(cap, multiplier * gain / weight)
            if improvement < deficit:
                lo = multiplier
            else:
                hi = multiplier

        corrections = [
            min(cap, hi * gain / weight)
            for gain, cap, weight in zip(gains, caps, weights)
        ]
        return corrections, hi / penalty

    def _barrier_deficits(self, temperature, low, high, rate):
        """Convert upper/lower discrete CBF inequalities into K deficits.

        h_upper = high - guard - T;  h_lower = T - low - guard.
        T_next is approximated by T + rate before action corrections.
        A positive deficit asks the QP to supply at least that much
        effective cooling (upper) or heating (lower), or pay soft slack.
        """
        upper = rate + self.BARRIER_ALPHA * (
            temperature - high + self.GUARD_BAND_K
        )
        lower = -rate + self.BARRIER_ALPHA * (
            low + self.GUARD_BAND_K - temperature
        )
        return upper, lower

    def _cooling_headroom(self, temperature, ahu, continuous):
        """Conservative cooling proxy: hot zone relative to V7 coil target.

        The coil setpoint is not guaranteed delivered air temperature.
        Require positive temperature headroom rather than assuming that
        raising the damper/fan always cools a zone.
        """
        index = ahu["supply"]
        if not self._valid_index(index, len(continuous)):
            return 0.0
        target_k = 295.0 + 10.0 * float(continuous[index])
        return max(0.0, temperature - target_k)

    def _cooling_cbf(self, temperature, high, deficit, zone, ahu, continuous):
        """Correct an overheated VAV, returning *severe* residual fan demand.

        The AHU fan boost is eligible only when:
        - the zone is beyond the severe excursion threshold;
        - the AHU coil setpoint has sufficient cooling headroom;
        - the VAV is near saturation after the local QP.
        """
        damper = zone["damper"]
        if not self._valid_index(damper, len(continuous)):
            return 0.0
        headroom = self._cooling_headroom(temperature, ahu, continuous)
        if headroom < self.MIN_COOLING_HEADROOM_K:
            return 0.0

        severe = temperature - high >= self.SEVERE_EXCURSION_K
        opening = self._fraction(continuous, damper)
        cap = min(
            self.SEVERE_FRACTION_CAP if severe else self.MILD_FRACTION_CAP,
            max(0.0, self.MAX_COMMANDED_FRACTION - opening),
        )
        gain = min(self.MAX_COOLING_GAIN,
                   self.DAMPER_GAIN_PER_K * headroom)
        penalty = 16.0 if severe else 4.0
        correction, _ = self._solve_soft_cbf_qp(
            deficit, [gain], [cap], [1.0], penalty
        )
        self._increase_fraction(continuous, damper, correction[0])

        # Shared fan pressure has an energy cost, so never activate it for
        # mild violations or for a VAV with ample remaining airflow range.
        after_opening = self._fraction(continuous, damper)
        if not severe or after_opening < self.FAN_DAMPER_THRESHOLD:
            return 0.0
        return max(0.0, deficit - gain * correction[0])

    def _heating_cbf(self, temperature, low, deficit, zone, continuous):
        """Increase reheat only; added airflow may instead cool the zone.

        V7 never uses the cooling-only AHU coil for heating. Without a
        model of post-reheat discharge temperature, increasing the VAV
        damper cannot safely be treated as positive heating authority.
        """
        valve = zone["valve"]
        if not self._valid_index(valve, len(continuous)):
            return

        severe = low - temperature >= self.SEVERE_EXCURSION_K
        opening = self._fraction(continuous, valve)
        cap = min(
            self.SEVERE_FRACTION_CAP if severe else self.MILD_FRACTION_CAP,
            max(0.0, self.MAX_COMMANDED_FRACTION - opening),
        )
        correction, _ = self._solve_soft_cbf_qp(
            deficit, [self.REHEAT_VALVE_GAIN], [cap], [1.0],
            16.0 if severe else 4.0,
        )
        self._increase_fraction(continuous, valve, correction[0])

    def _filter_zone(self, obs, zone, ahu, continuous, discrete, fallback):
        """Apply the relevant barrier and return severe cooling residual."""
        sensor = zone["temperature"]
        status = self._read_zone(obs, zone, fallback)
        if status is None:
            self._zone_history.pop(sensor, None)
            return 0.0

        temperature = status["temp"]
        low, high = self._zone_bounds(obs, zone, fallback)
        rate = self._temperature_rate(sensor, temperature)

        # Do not start equipment that nominal V7 has switched off.
        if discrete[ahu["run"]] == 0:
            return 0.0

        upper, lower = self._barrier_deficits(temperature, low, high, rate)
        if (upper > 0.0 and upper >= lower
                and temperature >= high - self.NEAR_BOUNDARY_K):
            return self._cooling_cbf(
                temperature, high, upper, zone, ahu, continuous
            )

        # Reheat works only if the nominal hot-water circuit is already on.
        if (lower > 0.0 and temperature <= low + self.NEAR_BOUNDARY_K
                and 0 <= self.water_run < len(discrete)
                and discrete[self.water_run] == 1):
            self._heating_cbf(temperature, low, lower, zone, continuous)
        return 0.0

    def _fan_cbf(self, deficit, ahu, continuous):
        """Optional shared AHU correction for persistent severe overheating."""
        pressure = ahu["pressure"]
        if deficit <= 0.0 or not self._valid_index(pressure, len(continuous)):
            return
        nominal_command = float(continuous[pressure])
        cap = min(
            self.MAX_FAN_ACTION_CORRECTION,
            max(0.0, 1.0 - nominal_command),
        )
        correction, _ = self._solve_soft_cbf_qp(
            deficit, [self.FAN_GAIN], [cap], [4.0], 12.0
        )
        if correction[0] > 0.0:
            continuous[pressure] = self._clamp(
                nominal_command + correction[0], -1.0, 1.0
            )

    def get_action(self, obs):
        """Run nominal V7 once, followed by bounded zone and fan QPs."""
        action = super().get_action(obs)
        observation = np.asarray(obs, dtype=np.float64).reshape(-1)
        fallback = self._schedule_fallback(observation)
        continuous = action["continuous_action"]
        discrete = action["discrete_action"]

        for ahu in self.ahus:
            worst_severe_cooling_deficit = 0.0
            for zone in ahu["zones"]:
                severe_deficit = self._filter_zone(
                    observation, zone, ahu, continuous, discrete, fallback
                )
                worst_severe_cooling_deficit = max(
                    worst_severe_cooling_deficit, severe_deficit
                )
            if discrete[ahu["run"]] == 1:
                self._fan_cbf(
                    worst_severe_cooling_deficit, ahu, continuous
                )

        # The V7 baseline already returns float32 and int32 arrays.
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return action


In [ ]:
# Official Kaggle evaluation gateway. Do not replace with a local CSV writer.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
